# Python и интернет

In [ ]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


## Зачем нам это все

Посмотрим на несколько знакомых ситуаций.

**Пример 1.** Вы вводите адрес сайта и через секунду видите страницу. За это время компьютер успел найти сервер, установить соединение и получить файлы. Если страница не открылась, нужно понять, на каком из этих шагов возникла проблема.

**Пример 2.** На одном компьютере сайт открывается в одном браузере и показывает предупреждение о безопасности в другом. Причина может быть в сертификате сайта и в том, каким удостоверяющим центрам доверяют браузеры. Ниже разберем, как устроена эта проверка.

**Пример 3.** Вам нужно собрать заголовки статей или сведения о книгах. Можно копировать их руками, а можно получить страницу питоном и достать нужные значения в список или словарь.

На занятии пройдем этот путь: найдем сервер по адресу сайта, разберемся с защищенным соединением, получим страницу через **requests** и извлечем данные через **BeautifulSoup**.

---

## Как устроен интернет

### Откуда берется страница

Когда вы открываете файл на своем компьютере, программа читает данные с его диска. Когда вы открываете сайт, нужные данные обычно находятся на другом компьютере - **сервере**. Ваш компьютер обращается к нему, получает ответ, а браузер показывает результат.

Например, при работе из дома цепочка будет выглядеть так:
- Ноутбук подключается по Wi-Fi к домашнему роутеру;
- Роутер связан с оборудованием интернет-провайдера, то есть компании, которая предоставляет вам доступ;
- Дальше данные могут пройти через оборудование нескольких компаний, прежде чем попадут на сервер с нужным сайтом.

Большие сообщения при передаче разбиваются на небольшие части - **пакеты**. В них есть данные и служебная информация, в том числе адрес назначения. Устройства по пути передают пакеты дальше в нужном направлении.

Браузер - программа для работы с веб-страницами. Но интернетом пользуются и другие программы: почта, мессенджеры, игры, а чуть позже - наш код на питоне. **Веб (World Wide Web, WWW)** - система страниц и других ресурсов, связанных адресами и ссылками. Это одна из систем, работающих через интернет.

### Клиент и сервер

В обмене данными удобно выделять две роли.

**Клиент** - программа, которая обращается за данными. Когда вы открываете сайт, клиентом является браузер. Когда скачиваете страницу питоном, клиентом становится ваша программа.

**Сервер** - программа, которая принимает обращения и отвечает на них. Например, получает просьбу показать статью, находит ее и отправляет клиенту. Словом «сервер» также называют компьютер, на котором такая программа работает. Поэтому в разговоре важно различать устройство и программу на нем.

```text
Ваш компьютер                         Компьютер с сайтом

Браузер    ─── запрос: дайте статью ──→ Программа-сервер
           ←── ответ: вот статья ─────
```

- **Запрос (request)** - сообщение клиента серверу.
- **Ответ (response)** - сообщение сервера клиенту. В ответе приходят не только сами данные, но и сведения о том, удалось ли выполнить запрос.

Клиент и сервер должны понимать одинаковые правила обмена. Такие правила называются **протоколом**. Например: как указать адрес нужной страницы, как сообщить об ошибке и как отделить служебную информацию от содержимого.

Для работы с сайтами используется протокол **HTTP (HyperText Transfer Protocol)**. По нему можно передавать текст страниц, картинки, файлы и данные в других форматах. Защищенный вариант HTTP называется **HTTPS**; к его защите мы скоро вернемся.

Клиент и сервер - роли в конкретном обмене, а не два обязательных типа компьютеров. Программу-сервер можно запустить и на обычном ноутбуке.

### Как компьютер находит другой компьютер: IP и DNS

Чтобы отправить данные другому устройству, нужен его **IP-адрес**. Например, `192.168.1.10`. По этому адресу сетевое оборудование направляет данные к получателю.

Внутри домашней или университетской сети используют внутренние адреса. Поэтому `192.168.1.10` в двух разных квартирах может относиться к разным устройствам. А адрес `127.0.0.1` означает «этот же компьютер»: он пригодится, когда будем запускать собственный сервер.

У сайтов есть **доменные имена**, например `example.com` и `habr.com`. Их удобнее запоминать, чем числовые адреса. Чтобы обратиться к сайту по имени, компьютер сначала узнает его IP-адрес:

1. Вы вводите `habr.com` в браузере.
2. Компьютер обращается к **DNS (Domain Name System)** - системе, которая позволяет узнать адрес по имени.
3. DNS возвращает адрес, по которому можно связаться с сервером.

Ответ DNS на некоторое время сохраняется, чтобы не спрашивать одно и то же при каждом обращении. Такое временное сохранение называется **кешированием**.

У одного имени может быть несколько IP-адресов, а на одном адресе могут работать разные сайты. Поэтому имя сайта и его IP-адрес не всегда взаимозаменяемы.

### Как попасть в нужную программу: порт

На одном компьютере могут одновременно работать сайт, почта и другие программы. Чтобы обратиться к нужной программе, вместе с IP-адресом используют **порт** - номер, на котором она принимает соединения. IP-адрес можно сравнить с адресом здания, а порт - с номером кабинета.

Для HTTP обычно используют порт **80**, для HTTPS - **443**. Поэтому в адресе `https://example.com` порт можно не писать: браузер выберет 443. Адрес `https://example.com:443` означает то же самое.

Программа «слушает порт», если готова принимать на нем соединения. Если она не запущена или доступ закрыт настройками сети, соединиться не получится.

### Из чего состоит адрес страницы

Полный адрес страницы или другого ресурса называется **URL (Uniform Resource Locator)**. Разберем пример:

```text
https://example.com:443/news/index.html?topic=python&page=2#comments
```

| Часть | Значение | Зачем нужна |
| --- | --- | --- |
| Схема | `https` | Как обращаться к ресурсу |
| Доменное имя | `example.com` | К какому сайту обратиться |
| Порт | `443` | С какой программой соединиться |
| Путь | `/news/index.html` | Какой ресурс запросить |
| Параметры | `topic=python&page=2` | Какие условия передать серверу |
| Фрагмент | `comments` | К какой части страницы перейти |

Порт можно опустить, если он стандартный для выбранной схемы. Путь не обязательно соответствует файлу на диске: сервер может формировать страницу программой.

**Параметры** начинаются после `?`. Они записываются парами `имя=значение`, разделенными символом `&`. Например, `page=2` может означать вторую страницу результатов. Названия и смысл параметров определяет сайт.

**Фрагмент** начинается после `#`. Браузер использует его, например, чтобы прокрутить страницу до комментариев. Серверу фрагмент не передается.

Разберем адрес средствами питона. Функция `urlsplit` разделит URL на части, а `parse_qs` превратит параметры в словарь. Здесь мы только разбираем строку, запрос к сайту пока не отправляется.

In [ ]:
from urllib.parse import urlsplit, parse_qs

url = 'https://example.com:443/news/index.html?topic=python&page=2#comments'
parts = urlsplit(url)

print('Схема:', parts.scheme)
print('Домен:', parts.hostname)
print('Порт:', parts.port)
print('Путь:', parts.path)
print('Параметры:', parse_qs(parts.query))
print('Фрагмент:', parts.fragment)

Схема: https
Домен: example.com
Порт: 443
Путь: /news/index.html
Параметры: {'topic': ['python'], 'page': ['2']}
Фрагмент: comments


### Что происходит после нажатия Enter

Соберем весь путь для адреса `https://example.com/news/`:

1. **Браузер разбирает адрес.** Нужен сайт `example.com`, путь `/news/`, соединение по HTTPS и порт 443.
2. **Компьютер узнает IP-адрес через DNS.** Если ответ уже сохранен, использует его.
3. **Устанавливается соединение с сервером.** Данные передаются между клиентом и сервером.
4. **Проверяется защищенное соединение.** Клиент проверяет сертификат сервера, стороны договариваются о защите обмена.
5. **Браузер отправляет HTTP-запрос и получает ответ.** В ответе могут быть страница, перенаправление или сообщение об ошибке.
6. **Браузер показывает страницу.** Разбирает HTML, получает нужные картинки, оформление и программы для работы страницы.

Одна страница часто требует нескольких запросов. Например, текст и картинки загружаются отдельно, иногда с разных сайтов. При повторном открытии браузер может использовать сохраненные файлы и уже установленное соединение.

Если сайт не открывается, проверять нужно конкретный этап: удалось ли узнать адрес, соединиться с сервером, проверить сертификат и получить ответ.

**Проверьте себя:** в адресе `https://example.com/books/?author=Толстой&page=3#reviews` найдите домен, путь, параметры и фрагмент. Какой порт будет использован? Что не попадет в запрос серверу?

---

## HTTPS и сертификаты

### Зачем защищать соединение

Данные между компьютером и сервером проходят через чужое оборудование. Нужно защитить их от чтения и изменения, а также убедиться, что на другом конце соединения нужный сервер.

Для этого используется **TLS (Transport Layer Security)** - протокол защищенного обмена. Он решает три задачи:

- **Шифрование** - посторонний не должен прочитать перехваченные данные.
- **Проверка целостности** - изменение данных по дороге должно быть обнаружено.
- **Проверка подлинности** - клиент проверяет, что сервер имеет право представлять имя из адреса сайта.

**HTTPS - HTTP через соединение, защищенное TLS.** Клиент по-прежнему отправляет запросы и получает ответы, но их передача защищена.

Чтобы проверить сервер, клиент использует его **сертификат**. Без такой проверки можно установить зашифрованное соединение с чужим сервером, который выдает себя за нужный.

В статьях часто встречается название «SSL-сертификат». Это старое название: современные соединения используют TLS.

Подробнее: [TLS на MDN](https://developer.mozilla.org/en-US/docs/Web/Security/Defenses/Transport_Layer_Security).

### Что такое сертификат и ключи

У сервера есть **закрытый ключ** - секретное числовое значение. С его помощью можно создать **цифровую подпись** для сообщения. Подпись позволяет проверить, что сообщение подписано владельцем ключа и не изменилось после подписания.

Для проверки используется другой, **открытый ключ**, который связан с закрытым. Его можно передавать всем: он позволяет проверить подпись, но не дает возможности подписывать сообщения от имени сервера. Закрытый ключ при проверке никому не отправляют.

При подключении сервер использует закрытый ключ, чтобы подтвердить, что он его владелец. Клиент проверяет это по открытому ключу. Но откуда клиент знает, что этот открытый ключ относится к нужному сайту?

Для этого сервер предъявляет **сертификат** - цифровой документ, в котором записаны:

- имена сайтов, для которых он выдан;
- открытый ключ сервера;
- начало и конец срока действия;
- сведения о выдавшем центре и его цифровая подпись.

Так сертификат связывает имя сайта с открытым ключом. При установлении соединения клиент проверяет сертификат, а стороны договариваются о ключах для шифрования дальнейшего обмена.

Самостоятельно написать документ с именем `example.com` может кто угодно. Поэтому важно, кто подписал сертификат и доверяет ли ему клиент.

### Кто подтверждает сертификат: цепочка доверия

Сертификаты сайтов выпускают **удостоверяющие центры (Certificate Authority, CA)**. Центр проверяет право заявителя использовать доменное имя и подписывает сертификат своим ключом. Некоторые виды сертификатов также требуют проверки организации.

У системы, браузера или библиотеки есть **хранилище доверенных корневых сертификатов (trust store)**. В него заранее включены сертификаты центров, которым этот клиент доверяет. Такие центры называют **корневыми**.

Обычно корневой центр подписывает сертификат промежуточного центра, а тот - сертификат сайта. Получается **цепочка доверия**:

```text
Корневой центр → Промежуточный центр → Сертификат сайта
      ↑
   уже есть в доверенном хранилище клиента
```

Стрелка означает «подписывает». Сервер присылает сертификат сайта и обычно сертификаты промежуточных центров. Клиент проверяет подписи по открытым ключам и доходит по цепочке до корня, который уже есть в его доверенном хранилище.

Просто прислать еще один корневой сертификат недостаточно. Если бы сервер мог сам назначать себе доверенный корень, такая проверка не имела бы смысла.

### Что может пойти не так

При проверке сертификата клиенту важны имя сайта, срок действия и цепочка доверия.

| Проблема | Что это означает |
| --- | --- |
| Истек срок | Текущая дата позже окончания действия сертификата |
| Сертификат еще не действует | Текущая дата раньше начала действия; возможно, сбились часы устройства |
| Другое имя | Сертификат выдан не для имени из адреса |
| Не строится доверенная цепочка | Не хватает промежуточного сертификата или клиент не доверяет корню |
| Сертификат отозван | Выдавший центр досрочно прекратил его действие |

**Истечение срока и отзыв - разные события.** Срок заканчивается в указанную дату. Отзыв происходит раньше: например, если закрытый ключ попал к посторонним. Клиенты могут по-разному получать и проверять сведения об отзыве.

Сертификат подтверждает связь ключа с именем сайта, но не честность его владельца: у мошеннического магазина тоже может быть действующий сертификат.

### Кейс: почему летом 2026 года сайты открывались не у всех

Летом 2026 года проблемы с сертификатами затронули сайты ФНС и ряда банков.

**Июнь-июль.** GlobalSign начал отзывать сертификаты ряда российских организаций. 20 июля CNews сообщил о недоступности единого личного кабинета налогоплательщика `elk.nalog.gov.ru`, связав проблему с отзывом сертификата GlobalSign. По сообщению издания, основной сайт ФНС продолжал работать. [CNews, 20 июля 2026](https://www.cnews.ru/news/top/2026-07-20_u_rossijskoj_nalogovoj_otobrali).

**Август.** РБК сообщил, что сайты ряда банков, включая ВТБ и Альфа-Банк, перестали открываться в зарубежных браузерах после перехода на сертификаты Национального удостоверяющего центра Минцифры. В «Яндекс Браузере», где было встроено доверие к российским корневым сертификатам, сайты открывались. [РБК, 3 августа 2026](https://www.rbc.ru/technology_and_media/03/08/2026/6a70c0239a7947e16e3c1c7d).

В случае с отзывом клиент отклоняет прежний сертификат, если получил и учитывает сведения об отзыве. После замены сертификата нужно проверить уже другую цепочку: до корня нового центра.

У браузеров наборы доверенных корней могут различаться. В августовском случае в «Яндекс Браузере» нужный корень был, а в других браузерах его могло не быть. Сервер продолжал работать, но часть клиентов не могла установить с ним защищенное соединение.

Это не истечение срока: прежний сертификат отозвали, а новый был выпущен другим центром.

### А если клиент - датчик или терминал?

К серверу может обращаться не только браузер. Например, платёжный терминал отправляет сведения об оплате, а датчик температуры - результаты измерений. Такие подключенные устройства относят к **интернету вещей (Internet of Things, IoT)**.

Датчик, установленный несколько лет назад, может до сих пор использовать старый набор доверенных сертификатов. Если сервер перейдет на сертификат другого центра, датчик может перестать передавать данные. Поэтому при установке таких устройств нужно предусмотреть и обновление их программного обеспечения.

**Небольшая практика.** Откройте `https://example.com` в браузере:

1. Через информацию о соединении рядом с адресом откройте просмотр сертификата.
2. Найдите имя сайта, выдавший центр, начало и конец срока действия.
3. Посмотрите цепочку сертификатов.

Объясните своими словами:

- Почему сервер не может сам назначить свой корень доверенным для всех клиентов?
- Почему правильные дата и время важны для HTTPS?
- Чем отзыв отличается от истечения срока?

---

## Получаем страницу через Python: requests

Из электронных газет, блогов и форумов можно собирать тексты для анализа. Например, собрать корпус или найти статьи о компании и оценить тональность сообщений. Сначала нужно получить сами страницы.

Для этого используем **requests** - библиотеку, которая отправляет запросы и получает ответы серверов.

Скачаем главную страницу Хабра. Питон отправит запрос так же, как браузер при открытии адреса.

In [ ]:
import requests

In [ ]:
response = requests.get('https://habr.com/ru/', timeout=10)
response

<Response [200]>

### Что мы только что сделали

`requests.get(...)` отправляет запрос с методом **GET** - просит сервер вернуть ресурс по указанному адресу. Метод определяет действие: например, POST часто используется для отправки формы.

Аргумент `timeout=10` ограничивает ожидание соединения и очередных данных. Это не общий предел на всё скачивание: пока сервер продолжает присылать данные, запрос может длиться дольше десяти секунд.

В `response` лежит **объект ответа**: содержимое страницы и сведения о запросе. Через точку можно получить отдельное значение, например `response.status_code`, или вызвать метод с круглыми скобками - `response.raise_for_status()`.

При выводе `response` мы видим краткую запись `<Response [200]>`. Число - **код статуса**, сообщение сервера о результате запроса.

| Статус | Смысл |
| --- | --- |
| 200 OK | Запрос успешно обработан |
| 301, 302 | Перенаправление на другой адрес; requests для GET обычно переходит автоматически |
| 403 Forbidden | Сервер отказывает в доступе |
| 404 Not Found | Ресурс по этому адресу не найден |
| 429 Too Many Requests | Запросов слишком много |
| 500 Internal Server Error | Ошибка при обработке запроса на сервере |

После статуса проверяют содержимое: с кодом 200 может прийти и нужная страница, и форма входа.

In [ ]:
response.status_code

200

In [ ]:
response.raise_for_status()
print('HTTP-ошибки 4xx/5xx нет')

HTTP-ошибки 4xx/5xx нет


`raise_for_status()` вызывает **исключение** для статусов 4xx и 5xx. Исключение сообщает об ошибке и останавливает текущую ячейку, если его не обработать.

Сам `requests.get()` не вызывает исключение при ответе 404 или 500: сервер ответил, но сообщил, что запрос не удалось выполнить.

В `response.text` находится содержимое ответа как строка. Выведем первые 700 символов:

In [ ]:
print(response.text[:700])

<!DOCTYPE html>
<html lang="ru">

  <head>
    <title>Публикации &#x2F; Моя лента &#x2F; Хабр</title>
<link rel="image_src" href="/img/habr_ru.png" data-hid="2a79c45">
<link href="https://habr.com/ru/feed/" rel="canonical" data-hid="e3fa780">
<meta itemprop="image" content="/img/habr_ru.png">
<meta property="og:image" content="/img/habr_ru.png">
<meta property="og:image:width" content="1200">
<meta property="og:image:height" content="630">
<meta property="aiturec:image" content="/img/habr_ru.png">
<meta name="twitter:image" content="/img/habr_ru.png">
<meta property="vk:image" content="/img/habr_ru.png?format=vk">
<meta property="fb:app_id" content="444736788986613">
<meta property="fb:pages


### Что еще лежит в ответе

В ответе видны конструкции `<html>`, `<head>` и `<title>`. Это **HTML** - текстовое описание структуры страницы. Браузер использует его, чтобы показать страницу на экране.

Кроме содержимого у ответа есть **заголовки (headers)** - служебные пары «имя - значение». Например, `Content-Type` сообщает, что пришло: HTML, изображение или другой тип данных. Эти заголовки относятся ко всему ответу, а не к названиям статей.

Текст передается последовательностью **байтов** - единиц цифровых данных. Чтобы превратить их в символы, нужна **кодировка** - правило соответствия между числовыми значениями и текстом. Если кодировка выбрана неверно, вместо кириллицы можно увидеть нечитаемые символы.

В ответе доступны:

- `response.url` - конечный адрес;
- `response.encoding` - кодировка, которую использует requests;
- `response.headers` - заголовки ответа;
- `response.content` - исходные байты;
- `response.text` - строка после декодирования, то есть превращения байтов в символы.

Для разбора HTML дальше будем использовать `response.text`.

In [ ]:
print('Конечный адрес:', response.url)
print('Кодировка:', response.encoding)
print('Тип содержимого:', response.headers.get('Content-Type'))
print('Первые байты:', response.content[:60])
print('Первые символы:', response.text[:60])

Конечный адрес: https://habr.com/ru/feed/
Кодировка: utf-8
Тип содержимого: text/html; charset=utf-8
Первые байты: b'<!DOCTYPE html>\n<html lang="ru">\n\n  <head>\n    <title>\xd0\x9f\xd1\x83\xd0\xb1'
Первые символы: <!DOCTYPE html>
<html lang="ru">

  <head>
    <title>Публик


Заголовки доступны по имени, как значения словаря. `response.headers.get('Content-Type')` возвращает тип содержимого или `None`, если заголовка нет. `dict(response.headers)` дает обычный словарь со всеми заголовками ответа.

У отправленного запроса тоже есть заголовки: например, в них клиент сообщает название приложения. Посмотрим их в `response.request.headers` и сравним с заголовками ответа:

In [ ]:
print('Метод нашего запроса:', response.request.method)
print('Адрес нашего запроса:', response.request.url)
print('Заголовки нашего запроса:', dict(response.request.headers))
print('Тип полученного ответа:', response.headers.get('Content-Type'))

Метод нашего запроса: GET
Адрес нашего запроса: https://habr.com/ru/feed/
Заголовки нашего запроса: {'User-Agent': 'python-requests/2.32.5', 'Accept-Encoding': 'gzip, deflate, br', 'Accept': '*/*', 'Connection': 'keep-alive'}
Тип полученного ответа: text/html; charset=utf-8


При перенаправлении `response.url` показывает конечный адрес, а `response.history` содержит промежуточные ответы. Так можно заметить, что вместо статьи сервер отправил нас на страницу входа.

Описание атрибутов и методов есть в [документации requests](https://requests.readthedocs.io/en/latest/api/).

### Передаем параметры запроса

Параметры после `?` задают условия запроса: поисковую строку, номер страницы, фильтр. В requests их передают словарем через **`params`**. Библиотека сама подготовит URL с учетом пробелов, кириллицы и специальных символов.

Передадим поисковую строку и номер страницы сервису **httpbin**. Это учебный сервис: по адресу `/get` он возвращает сведения о полученном запросе. В ответе посмотрим, дошли ли наши параметры до сервера.

Ответ придет в формате JSON. Метод `echo_response.json()` разбирает этот текст в объекты питона. В данном ответе ключ `args` содержит параметры запроса.

In [ ]:
params = {'q': 'питон и интернет', 'page': 2}

echo_response = requests.get('https://httpbin.org/get', params=params, timeout=10)
echo_response.raise_for_status()

print('Адрес:', echo_response.url)
print('Тип ответа:', echo_response.headers.get('Content-Type'))
print('Параметры, которые получил сервер:', echo_response.json()['args'])

Адрес: https://httpbin.org/get?q=%D0%BF%D0%B8%D1%82%D0%BE%D0%BD+%D0%B8+%D0%B8%D0%BD%D1%82%D0%B5%D1%80%D0%BD%D0%B5%D1%82&page=2
Тип ответа: application/json
Параметры, которые получил сервер: {'page': '2', 'q': 'питон и интернет'}


В URL пробелы и кириллица записаны специальным образом, но сервер получил исходный текст. Число `2` стало строкой: параметры URL передаются текстом.

Пароли через GET-параметры передавать не стоит. Адрес может сохраниться в истории браузера и журналах сервера. HTTPS защищает передачу по сети, но не предотвращает такое сохранение.

### Как понять, что именно сломалось

Если сервер вернул 404, HTTP-ответ уже получен. Если не удалось узнать адрес или проверить сертификат, до получения страницы дело не дошло.

| Что видим в requests | Где возникла проблема |
| --- | --- |
| `ConnectionError` | Не удалось соединиться: возможна проблема с DNS, сетью или сервером |
| `SSLError` | Ошибка защищенного соединения, в том числе проверки сертификата |
| `Timeout` | Не дождались соединения или очередных данных |
| `HTTPError` после `raise_for_status()` | Сервер ответил статусом 4xx или 5xx |
| Статус 200, но нужных данных нет | Нужно проверить, какая страница пришла |

Проверка сертификатов в requests включена по умолчанию. При ошибке проверяют имя сайта в URL, часы устройства, срок действия и цепочку сертификатов. Если тот же адрес открывается в браузере, стоит сравнить, какие центры доверены браузеру и питону.

`verify=False` отключает проверку подлинности сервера. Причину ошибки это не устраняет. Подробнее: [проверка сертификатов в requests](https://requests.readthedocs.io/en/latest/user/advanced/#ssl-cert-verification).

У учебного сайта `expired.badssl.com` намеренно просрочен сертификат. В следующей ячейке ожидаем `SSLError`. Конструкция `try` / `except` перехватывает эту ошибку и позволяет продолжить выполнение.

In [ ]:
try:
    bad_response = requests.get('https://expired.badssl.com/', timeout=10)
    bad_response.raise_for_status()
except requests.exceptions.SSLError:
    print('SSLError: проверка сертификата не прошла. HTTP-страницу мы не получили.')

SSLError: проверка сертификата не прошла. HTTP-страницу мы не получили.


In [ ]:
bad_response = requests.get('https://expired.badssl.com/', timeout=10)
bad_response.raise_for_status()

SSLError: HTTPSConnectionPool(host='expired.badssl.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1032)')))

При прямом соединении пример дает ошибку сертификата. В сети организации результат может отличаться, если защитное ПО заново устанавливает HTTPS-соединение от своего имени.

**Проверьте себя:** если проверка сертификата не прошла, получили ли мы HTML страницы?

---

## HTML: что именно прислал сервер

### Страница как текст

В браузере страница выглядит как заголовки, абзацы, ссылки и картинки. Сервер часто передает ее как текст с пометками о структуре. Такая запись называется **HTML (HyperText Markup Language)** - язык разметки гипертекста. **Гипертекст** - текст со ссылками на другие документы и разделы.

HTML указывает, где заголовок, абзац или ссылка. За оформление и поведение отвечают другие средства:

| Что | За что отвечает | Пример |
| --- | --- | --- |
| HTML | Структура и содержимое | Заголовок и два абзаца |
| CSS | Оформление | Крупный синий заголовок |
| JavaScript | Программа в браузере | Загрузка новых записей по кнопке |

Посмотрим на учебную страницу, записанную строкой питона. В ней две новости с придуманными заголовками и авторами.

In [ ]:
html_page = """<!DOCTYPE html>
<html lang="ru">
  <head>
    <meta charset="utf-8">
    <title>Новости семинара</title>
  </head>
  <body>
    <h1 id="top">Питон и интернет</h1>
    <p class="intro">Сегодня учимся доставать <b>данные</b> из HTML.</p>
    <!-- Комментарий: этот текст не показывается на странице -->
    <article class="post" id="post-1">
      <h2><a href="/news/1">Открылась запись на семинар</a></h2>
      <p class="author">Анна</p>
      <time datetime="2026-11-01">1 ноября</time>
    </article>
    <article class="post" id="post-2">
      <h2><a href="/news/2">Новая подборка материалов</a></h2>
      <p class="author">Мария</p>
      <time datetime="2026-11-02">2 ноября</time>
    </article>
    <a href="#top">Наверх</a>
    <br>
  </body>
</html>"""

### Как читать эту запись: теги и атрибуты

**Тег** - пометка в HTML, например `<p>` в начале абзаца и `</p>` в конце. Вся запись `<p>Текст абзаца</p>` называется **элементом**. Внутри могут находиться текст и другие элементы.

**Атрибут** - дополнительное свойство элемента. В записи `<a href="/news/1">Новость</a>` тег `a` создает ссылку, атрибут `href` хранит адрес `/news/1`, а текст `Новость` увидит пользователь. Когда будем собирать ссылки, адрес и видимый текст понадобятся отдельно.

У многих элементов есть открывающий и закрывающий теги. У `br`, `img` и `meta` закрывающих тегов нет: эти элементы не содержат вложенного содержимого.

В нашей странице используются:

- `<!DOCTYPE html>` - обозначение современного HTML;
- `html` - весь документ; `lang="ru"` указывает язык;
- `head` - служебные сведения; `title` задает название вкладки;
- `body` - содержимое страницы;
- `h1` и `h2` - заголовки разных уровней;
- `article` - блок одной новости;
- `p` - абзац, `b` - выделенный фрагмент, `a` - ссылка;
- `time` - дата для человека; `datetime` содержит запись для программы.

Название вкладки в `title` и главный заголовок в `h1` могут различаться. Комментарий `<!-- ... -->` виден в исходном коде, но не показывается как текст страницы.

Еще часто встречаются **`div` и `span`** - контейнеры для группировки содержимого. Чтобы понять, что в них находится, смотрят на вложенные элементы и атрибуты.

### Классы, id и вложенность

Чтобы находить и оформлять элементы, им дают обозначения:

- **`class`** объединяет похожие элементы. У обеих новостей класс `post`. Название выбирает автор сайта; у элемента может быть несколько классов через пробел.
- **`id`** обозначает один элемент и должен быть уникальным в документе. У первой новости `id="post-1"`, у второй - `id="post-2"`. Ссылка `href="#top"` ведет к элементу с `id="top"`.

Вложенность элементов можно представить деревом:

```text
html
├── head
│   └── title
└── body
    ├── h1
    ├── p
    │   └── b
    ├── article (первая новость)
    │   ├── h2
    │   │   └── a
    │   ├── p (автор)
    │   └── time
    └── article (вторая новость)
        └── ...
```

У каждой новости свой `article`, а внутри - заголовок, автор и дата. Поэтому удобнее сначала найти нужный `article` и искать остальные элементы уже в нем. Если собирать заголовки и авторов по всей странице отдельно, можно их перепутать: например, у одной новости автор не указан.

Ссылка `/news/1` **относительная**: чтобы получить полный адрес, нужно знать адрес исходной страницы. В **полном адресе** `https://example.com/news/1` уже есть схема и имя сайта.

**Проверьте себя:** найдите в `html_page` название вкладки, заголовок второй новости, автора и дату для программы. Что находится в тексте элементов, а что - в атрибутах?

### Почему в браузере есть данные, а в requests их нет

Сервер может подготовить начальный HTML двумя способами:

- Отдать готовый файл. Это **статическая страница**.
- Сформировать HTML программой при запросе: например, подставить имя пользователя и результаты поиска. Это **динамическое формирование на сервере**. Клиент получает результат, а не код программы сервера.

После получения HTML браузер может выполнить **JavaScript** - программу, которая добавляет элементы или загружает данные отдельными запросами. Так, например, появляются новые записи при прокрутке ленты.

**requests и BeautifulSoup не выполняют JavaScript.** Поэтому данные, появившиеся после его запуска, могут отсутствовать в `response.text`.

Чтобы найти источник данных, откройте инструменты разработчика браузера через меню или пункт «Исследовать элемент»:

- **Network / «Сеть»** показывает запросы и ответы. Откройте вкладку и обновите страницу, чтобы увидеть загрузку.
- **Elements / «Элементы»** показывает текущее дерево страницы, в том числе изменения JavaScript. Такое представление документа называют **DOM**.
- **Исходный код страницы** показывает первоначальный HTML.